# Phân khúc Khách hàng Online Retail bằng K-Means

## Mục tiêu
1. Xây dựng mô hình **K-Means** bằng scikit-learn trên đặc trưng RFM dùng chung của nhóm.
2. Tìm số cụm tối ưu bằng **Elbow Method** và **Silhouette Score** (kèm Davies-Bouldin, Calinski-Harabasz và độ ổn định bootstrap).
3. Trực quan hóa các cụm khách hàng.
4. Phân tích đặc điểm từng cụm.

## Hợp đồng dữ liệu (xem `docs/team_data_guide.md`)
- Đầu vào `X` = đúng 3 cột `Recency_scaled, Frequency_scaled, Monetary_scaled` (đã `log1p` + `StandardScaler` fit theo snapshot). **Không chuẩn hóa lại.**
- Chọn k, seed và tham số **chỉ trên Development** (lịch sử trước 01/06/2011) bằng chỉ số nội tại. Outcome 90 ngày của Development chỉ dùng để *mô tả* cụm sau khi đã chọn k.
- **Final** (lịch sử trước 01/09/2011): chạy lại đúng cấu hình đã chốt; notebook **không đọc outcome Final**.
- Đầu ra: `models/outputs/{development,final}/kmeans.csv` (`CustomerID,cluster`) + `kmeans.json` (tham số, thời gian chạy, metric).

Toàn bộ hàm nằm trong `kmeans.py`; notebook này gọi lại để hiển thị và diễn giải. Chạy `python kmeans.py` cho kết quả giống hệt.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data_cleaned").exists() and (ROOT / "Online-Retail-Customer-Segmentation" / "data_cleaned").exists():
    ROOT = ROOT / "Online-Retail-Customer-Segmentation"
for p_add in (str(ROOT), str(ROOT / "src")):
    if p_add not in sys.path:
        sys.path.insert(0, p_add)

import kmeans as km
from retail_segmentation.temporal_evaluation import load_phase, evaluate_simple_rules, evaluate_cluster_priority

DATA_DIR = ROOT / "data_cleaned"
OUTPUTS_ROOT = ROOT / "models" / "outputs"
FIGURES_DIR = ROOT / "reports" / "figures"
TABLES_DIR = ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

dev_rfm, dev_feat, dev_outcomes = load_phase(DATA_DIR, "development")
fin_rfm, fin_feat = km.load_model_inputs(DATA_DIR, "final")  # không đọc outcome Final
X_dev = km.feature_matrix(dev_feat)
print(f"Development: {len(dev_feat):,} khách | Final: {len(fin_feat):,} khách | X_dev shape = {X_dev.shape}")

## 1. Tổng quan dữ liệu đầu vào

In [ ]:
desc_rfm = dev_rfm[km.RFM_COLUMNS].describe(percentiles=[.25, .5, .75, .9, .99]).T
desc_scaled = dev_feat[km.FEATURE_COLUMNS].describe(percentiles=[.25, .5, .75, .9, .99]).T
display(pd.concat([desc_rfm, desc_scaled]).round(3))
print("Tương quan Pearson giữa 3 feature scaled:")
display(dev_feat[km.FEATURE_COLUMNS].corr().round(3))

**Nhận xét (TODO — điền sau khi chạy):**
- RFM gốc lệch phải mạnh (so mean với median, p99 với max); sau `log1p` + `StandardScaler`, mỗi feature có mean ≈ 0, std ≈ 1 → dùng thẳng, không chuẩn hóa lại.
- Frequency và Monetary tương quan cao (hệ số = …): hai trục này cùng kéo khoảng cách Euclid, nên K-Means sẽ phân tách chủ yếu theo "mức độ mua sắm".

## 2. Tìm số cụm tối ưu: Elbow + Silhouette

In [ ]:
search_df = km.search_k(X_dev, km.K_VALUES)  # mỗi k: inertia, Silhouette/DBI/CHI (evaluator chung), quy mô cụm, ARI bootstrap
search_df.to_csv(TABLES_DIR / "kmeans_k_search_development.csv", index=False)
elbow_k = km.find_elbow_k(search_df["k"], search_df["inertia"])
chosen_k, reason = km.choose_k(search_df, elbow_k)
display(search_df.drop(columns=["parameters"]).round(4))
print(reason)

In [ ]:
display(Image(km.plot_k_selection(search_df, elbow_k, chosen_k, FIGURES_DIR)))

**Phân tích (TODO):**
- **Elbow:** inertia giảm nhanh tới k = … rồi chậm dần; Kneedle chọn k = `elbow_k`.
- **Silhouette:** cao nhất ở k = … (thường k = 2 với RFM: chỉ tách "mua nhiều" và "mua ít", ít giá trị phân khúc). Trong khoảng k ∈ [3, 6], Silhouette cao nhất ở k = ….
- **Davies-Bouldin / Calinski-Harabasz:** có ủng hộ cùng k không?
- **Độ ổn định:** ARI bootstrap ở k đã chọn = … (ngưỡng 0.90); k lớn hơn có ARI giảm → cụm kém lặp lại.

In [ ]:
neighbours = [k for k in (chosen_k - 1, chosen_k, chosen_k + 1) if k >= 2]
display(Image(km.plot_silhouette_diagrams(X_dev, neighbours, FIGURES_DIR)))

### Quyết định số cụm (TODO)
Quy tắc cố định trong `choose_k`: trong k ∈ [3, 6], giữ các k có cụm nhỏ nhất ≥ 5% khách và ARI bootstrap ≥ 0.90; chọn Silhouette cao nhất; nếu k của Elbow chỉ kém < 0.02 Silhouette thì ưu tiên Elbow.

→ **k = …** vì … (chép `reason` ở trên và bổ sung nhận xét từ silhouette diagram: cụm nào có nhiều điểm silhouette âm?).

## 3. Trực quan hóa các cụm

In [ ]:
dev_labels, dev_centers, dev_runtime, dev_inertia = km.fit_chosen_model(X_dev, chosen_k)
dev_labeled = km.label_frame(dev_feat["CustomerID"], dev_labels, dev_rfm)
dev_profile = km.profile_clusters(dev_labeled, dev_outcomes)
names = km.name_clusters(dev_profile, dev_rfm)
dev_profile.insert(1, "cluster_name", dev_profile["cluster"].map(names))
dev_centroids = km.centroids_original_units(dev_centers, km.load_scaler(DATA_DIR, "development"))
dev_centroids.insert(1, "cluster_name", dev_centroids["cluster"].map(names))

pca_path, pca = km.plot_pca_clusters(X_dev, dev_labels, dev_centers, names, FIGURES_DIR)
display(Image(pca_path))
print("Hệ số tải (loadings) của PCA:")
display(pd.DataFrame(pca.components_, columns=km.FEATURE_COLUMNS, index=["PC1", "PC2"]).round(3))

**Diễn giải PCA (TODO):** PC1 (…% phương sai) chủ yếu là … (Frequency + Monetary − Recency → "giá trị/độ gắn bó"); PC2 (…%) chủ yếu là … (Recency → "mới/cũ"). Các cụm nằm dọc PC1 như bậc thang giá trị; có ranh giới chồng lấn ở … vì dữ liệu RFM liên tục, không có khoảng trống tự nhiên.

In [ ]:
display(Image(km.plot_centroid_heatmap(dev_centroids, names, FIGURES_DIR)))
display(Image(km.plot_rfm_boxplots(dev_labeled, FIGURES_DIR)))
display(dev_centroids.round(3))

## 4. Đặc điểm từng cụm

In [ ]:
dev_profile.to_csv(TABLES_DIR / "kmeans_development_cluster_profiles.csv", index=False)
dev_centroids.to_csv(TABLES_DIR / "kmeans_centroids_development.csv", index=False)
display(dev_profile)
display(Image(km.plot_cluster_summary(dev_profile, names, FIGURES_DIR)))

**Hồ sơ cụm (TODO — điền số thật):**

| Cụm | Tên | % khách | Median R / F / M | % Monetary | Mua lại 90 ngày | Gợi ý hành động |
|---|---|---:|---|---:|---:|---|
| 0 | … | … | … ngày / … HĐ / £… | … | … | Chăm sóc VIP, ưu đãi giữ chân |
| 1 | … | … | … | … | … | Bán chéo, tăng giá trị đơn |
| 2 | … | … | … | … | … | Kích hoạt lần mua thứ 2 |
| 3 | … | … | … | … | … | Win-back chi phí thấp |

Tỷ lệ mua lại 90 ngày là **liên hệ mô tả** trên Development, không phải tác động nhân quả của việc chăm sóc khách.

## 5. Tham khảo trên Development: chọn 15% khách theo cụm so với quy tắc đơn giản

In [ ]:
dev_label_df = pd.DataFrame({"CustomerID": dev_feat["CustomerID"], "cluster": dev_labels.astype(str)})
policies = pd.concat([
    evaluate_simple_rules(dev_rfm, dev_outcomes),
    pd.DataFrame([evaluate_cluster_priority(dev_rfm, dev_outcomes, dev_label_df, "kmeans")]),
], ignore_index=True)
display(policies.round(4))

**Nhận xét (TODO):** ở cùng 15% khách, ưu tiên theo cụm K-Means bao phủ …% doanh thu 90 ngày, so với top Monetary …% và RFM score …%. Bảng này chỉ để tham khảo, **không dùng để chọn lại k**. So sánh chính thức với các model khác chạy bằng `scripts/evaluate_temporal.py`.

## 6. Xuất nhãn Development và chạy lại trên Final

In [ ]:
selection = {"k_values": search_df["k"].tolist(), "elbow_k": elbow_k, "chosen_k": chosen_k, "rule": reason}
dev_metrics = km._metric_summary(X_dev, dev_labels, dev_runtime, dev_inertia)
km.export_labels(OUTPUTS_ROOT, "development", dev_feat["CustomerID"], dev_labels, dev_runtime, chosen_k, dev_metrics, selection)

X_fin = km.feature_matrix(fin_feat)
fin_labels, fin_centers, fin_runtime, fin_inertia = km.fit_chosen_model(X_fin, chosen_k)
fin_labeled = km.label_frame(fin_feat["CustomerID"], fin_labels, fin_rfm)
fin_profile = km.profile_clusters(fin_labeled)
fin_names = km.name_clusters(fin_profile, fin_rfm)
fin_profile.insert(1, "cluster_name", fin_profile["cluster"].map(fin_names))
fin_centroids = km.centroids_original_units(fin_centers, km.load_scaler(DATA_DIR, "final"))
fin_centroids.insert(1, "cluster_name", fin_centroids["cluster"].map(fin_names))
fin_profile.to_csv(TABLES_DIR / "kmeans_final_cluster_profiles.csv", index=False)
fin_centroids.to_csv(TABLES_DIR / "kmeans_centroids_final.csv", index=False)
fin_metrics = km._metric_summary(X_fin, fin_labels, fin_runtime, fin_inertia)
km.export_labels(OUTPUTS_ROOT, "final", fin_feat["CustomerID"], fin_labels, fin_runtime, chosen_k, fin_metrics, selection)

display(pd.DataFrame([dev_metrics, fin_metrics], index=["development", "final"]))
display(fin_profile)

**Kiểm tra cấu trúc trên Final (TODO):** cùng k, các cụm có giữ thứ tự và tên như Development không? Tỷ lệ khách và median RFM thay đổi thế nào? (Lịch sử trước hai snapshot khác nhau nên không kỳ vọng giống hệt.)

Bước tiếp theo của nhóm:
```bash
python scripts/evaluate_temporal.py --phase development --with-models
```
Chỉ chạy `--phase final --with-models` khi cả nhóm đã chốt model trên Development.

## 7. Hạn chế và Q&A

**Hạn chế của K-Means trên dữ liệu này**
- Giả định cụm dạng cầu, kích thước tương đương: dữ liệu RFM liên tục và lệch nên ranh giới cụm là cắt lát, không phải khoảng trống tự nhiên.
- Nhạy với ngoại lệ: vài khách chi tiêu rất lớn kéo tâm cụm; `log1p` giảm nhưng không loại bỏ ảnh hưởng này.
- Frequency và Monetary tương quan cao nên gần như được tính hai lần trong khoảng cách Euclid.
- Mọi khách đều bị gán cụm (không có noise như DBSCAN); khi so với DBSCAN cần đọc cùng quy mô cụm và tỷ lệ noise, không so điểm số máy móc.

**Q&A**
- *Vì sao không chọn k có Silhouette cao nhất tuyệt đối?* Với RFM, k = 2 thường có Silhouette cao nhất nhưng chỉ tách "mua nhiều / mua ít"; quy tắc giới hạn k ∈ [3, 6], yêu cầu quy mô và độ ổn định tối thiểu.
- *Vì sao nhãn ở Development và Final khớp nhau?* `relabel_by_value` đánh số cụm theo điểm giá trị tâm cụm (−R + F + M), và tên cụm đặt theo hồ sơ RFM chứ không theo số nhãn.
- *Có dùng outcome để chọn k không?* Không. Outcome Development chỉ dùng để mô tả sau khi đã chọn k; outcome Final không được đọc.